In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.18 Basis Sets, and the Error They Invent

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.18",
    title="Basis Sets, and the Error They Invent",
    blurb="Seventeen notebooks have represented electrons on grids and in plane "
    "waves. Quantum chemistry uses neither: it uses Gaussians glued to the nuclei, "
    "because a product of two Gaussians is a third Gaussian and every integral "
    "then closes in elementary functions. We build those integrals for one "
    "electron, watch a single Gaussian fail at the nuclear cusp, pay for a "
    "contraction, discover that a basis optimised for a molecule is wrong for a "
    "free atom, and then meet the pathology that belongs to atom-centred bases "
    "alone: a hydrogen atom whose energy falls when we hand it basis functions "
    "with no nucleus behind them.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

This volume has represented a wavefunction in exactly two ways. On a real-space
grid, from the radial machinery of [§8.1](many-electron-problem.ipynb) through the
exact laboratory of [§8.2](exact-laboratory.ipynb), Hartree–Fock in
[§8.3](hartree-fock-atoms.ipynb), the density-functional movement, and the
real-time propagation of [§8.16](tddft.ipynb). And in plane waves, from
[§8.10](plane-waves-pseudopotentials.ipynb) through the real silicon bands of
[§8.11](epm-band-structures.ipynb) and the optics of
[§8.15](optics-excitons.ipynb). Both are physicists' bases. Neither is what
quantum chemistry actually runs on. The overwhelming majority of molecular
calculations ever performed used a third representation: fixed Gaussian functions
glued to the nuclei, invented for one reason, that a product of two Gaussians
centred anywhere is a single Gaussian centred somewhere else, which turns every
multi-centre integral into elementary functions.

Arriving here *after* the grid and after plane waves is the honest order, because
the third representation is the only one of the three that invents an error the
other two cannot have. Its functions are attached to the nuclei, so moving a
nucleus moves the basis, and a system's description improves for reasons that have
nothing to do with its physics. That is basis set superposition error, and the
centrepiece of this notebook is a demonstration of it on a *single electron*: a
hydrogen atom whose computed energy drops when we place extra basis functions at a
distance with no nucleus behind them and no second electron anywhere.

Everything here stays at one electron with $s$-type primitives, and that
containment is deliberate. General Gaussian integral machinery, meaning two-electron
repulsion and arbitrary angular momentum, is a graduate one-semester project, and
every serious teaching resource hides it: the standard programming projects ship the
integrals in text files, and the well-known teaching packages compute them in
compiled code precisely so the student never has to. At one electron with $s$
functions there is nothing to hide. Every integral we need is a printable closed
form, so this notebook builds its own machinery from the first line and validates it
against numerical quadrature.

> **Conventions (this notebook).** Hartree atomic units, as everywhere in this
> volume: the exact hydrogen ground state is $-1/2$ Ha. Basis functions are
> *unnormalised* primitive $s$-Gaussians $\chi_p(\mathbf r) = \exp(-\alpha_p
> |\mathbf r - \mathbf A_p|^2)$, with normalisation carried by the generalised
> eigenvalue problem rather than by the functions. There is exactly one nucleus,
> charge $Z = 1$, at the origin, and exactly one electron. Matrices are assembled
> with `numpy` broadcasting, the generalised eigenproblem is solved by
> `scipy.linalg.eigh(H, S)`, and the independent integral checks use
> `scipy.integrate.dblquad` in spherical coordinates about the nucleus.
>
> **How to read the checks.** Every exercise closes with a `validate` call against
> something the computation did not assume: a closed-form integral, a two-dimensional
> quadrature, an exact eigenvalue, the variational bound, Kato's cusp condition. A ✓
> is strong evidence, not proof; a ✗ is a prompt to locate the discrepancy, which may
> be a genuine error, a convention mismatch, or a tolerance set too tightly, and not
> an automatic verdict.
>
> **Scope, stated plainly.** No two-electron integrals, no self-consistent field, no
> correlation: one electron from beginning to end. Polarisation and diffuse functions
> are *narrated and never computed*, because computing them needs $d$-functions and
> general angular momentum, and there is no canonical teaching demonstration of their
> importance that we could honestly reproduce. Counterpoise correction on a real
> bound dimer needs a quantum-chemistry package and is named as the horizon, not
> performed. The single-electron superposition-error demonstration in Exercise 5 has
> no published teaching precedent that we could find; the course is doing something
> the teaching literature does not, and says so rather than implying otherwise.
> For the general machinery, Szabo & Ostlund {cite}`szabo1996`, Ch. 3 and Appendix A,
> is the canonical reference; Martin {cite}`martin2004`, Ch. 12, places Gaussians
> beside the plane waves of [§8.10](plane-waves-pseudopotentials.ipynb).

## Theory in brief

### A basis turns a differential equation into a matrix problem

Expanding an unknown orbital in a fixed set of functions,
$\psi(\mathbf r) = \sum_q c_q \chi_q(\mathbf r)$, and demanding that the Rayleigh
quotient be stationary in the coefficients is the variational principle of
[§6.22](../06-quantum-mechanics/variational-method.ipynb) applied to a linear trial
function. Because the $\chi_q$ are in general *not orthogonal*, stationarity does
not produce an ordinary eigenvalue problem but the generalised one built from
scratch in [§0.5](../00-foundations/eigenvalues-svd.ipynb),

```{math}
:label: eq-gto-secular
\mathbf H\,\mathbf c = \varepsilon\,\mathbf S\,\mathbf c,
\qquad
H_{pq} = \langle \chi_p | \hat h | \chi_q \rangle,
\qquad
S_{pq} = \langle \chi_p | \chi_q \rangle .
```

The overlap matrix $\mathbf S$ is the entire difference from a textbook eigenvalue
problem, and it is where the trouble in this notebook eventually lives: as basis
functions crowd together, $\mathbf S$ approaches singularity and the expansion
stops being able to say anything new. In quantum chemistry
{eq}`eq-gto-secular` is the Roothaan equation; here, with one electron, $\hat h$ is
the whole Hamiltonian and the lowest $\varepsilon$ is the whole energy, so the
result is a strict variational upper bound on the exact ground state.

### Why Gaussians, when Slater functions are the physical ones

The hydrogenic orbitals of
[§6.17](../06-quantum-mechanics/hydrogen-atom.ipynb) decay as $e^{-\zeta r}$ and
have a cusp at the nucleus, so the physically honest basis function is the Slater
orbital $e^{-\zeta r}$. Its integrals over several centres, however, have no
elementary closed form. Boys observed in 1950 (S. F. Boys, *Proc. R. Soc. A* **200**,
542) that the physically *dishonest* choice repairs the arithmetic completely,
because the product of two Gaussians on different centres is one Gaussian on a third
centre. Writing the primitive as

```{math}
:label: eq-gto-primitive
\chi_p(\mathbf r) = \exp\!\big(-\alpha_p\,|\mathbf r - \mathbf A_p|^2\big),
```

a short completion of the square gives the Gaussian product theorem, the identity on
which the whole of quantum chemistry rests,

```{math}
:label: eq-gto-product
e^{-\alpha|\mathbf r - \mathbf A|^2}\,e^{-\beta|\mathbf r - \mathbf B|^2}
= K\,e^{-p\,|\mathbf r - \mathbf P|^2},
\qquad
p = \alpha + \beta,
\quad
\mu = \frac{\alpha\beta}{p},
\quad
\mathbf P = \frac{\alpha\mathbf A + \beta\mathbf B}{p},
\quad
K = e^{-\mu\,|\mathbf A - \mathbf B|^2}.
```

A two-centre integral has become a one-centre integral. Szabo & Ostlund
{cite}`szabo1996`, Appendix A, carries the same completion of the square through to
the general two-electron case; here we need only the three one-electron integrals.

### The three integrals, in closed form

With {eq}`eq-gto-product` in hand the overlap is a plain Gaussian integral, the
kinetic integral follows from differentiating it twice, and the nuclear attraction
needs one extra ingredient because $1/|\mathbf r - \mathbf C|$ is not a polynomial.
For a nucleus of charge $Z$ at $\mathbf C$ the three matrices are

```{math}
:label: eq-gto-integrals
\begin{aligned}
S_{pq} &= \Big(\frac{\pi}{p}\Big)^{3/2} K, \\[2pt]
T_{pq} &= \mu\,\big(3 - 2\mu\,|\mathbf A_p - \mathbf A_q|^2\big)\,S_{pq}, \\[2pt]
V_{pq} &= -\frac{2\pi Z}{p}\,K\,F_0\!\big(p\,|\mathbf P - \mathbf C|^2\big),
\end{aligned}
```

with $p$, $\mu$, $\mathbf P$ and $K$ read off {eq}`eq-gto-product`. The extra
ingredient is the lowest Boys function, which is what the Coulomb singularity leaves
behind after the angular integration,

```{math}
:label: eq-gto-boys
F_0(x) = \int_0^1 e^{-x t^2}\,dt
       = \frac{1}{2}\sqrt{\frac{\pi}{x}}\;\operatorname{erf}\big(\sqrt{x}\big),
\qquad F_0(0) = 1 .
```

That is the entire machinery of this notebook: three formulas and an error function.
The same three appear as a worked problem in ETH Zürich's *Computational Quantum
Physics* course (Exercise Sheet 8, Problem 8.1, "re-solving the hydrogen atom in the
GTO basis via variational method"), which prints $S$, $T$ and $V$ exactly as above;
it is Thijssen, *Computational Physics*, Ch. 3.2.2, in problem-sheet dress, and
Filot, *Elements of Electronic Structure Theory*, poses the same calculation as
Exercise 3.10.

### Contraction, and the scaling factor nobody mentions

A single Gaussian is a poor $1s$ orbital, and Exercise 2 measures exactly how poor.
The standard repair is to fit a Slater orbital with a *fixed* linear combination of
Gaussians and then treat that combination as one basis function. STO-3G, the
minimal basis of Hehre, Stewart and Pople (*J. Chem. Phys.* **51**, 2657, 1969), is
the three-term least-squares fit,

```{math}
:label: eq-gto-contraction
\phi(\mathbf r) = \sum_{i=1}^{3} d_i\,N_i\,e^{-\alpha_i r^2},
\qquad
N_i = \Big(\frac{2\alpha_i}{\pi}\Big)^{3/4},
```

where the $N_i$ normalise the primitives (the tabulated $d_i$ assume normalised
primitives) and the $d_i$ are frozen: the variational problem sees one function, not
three. What is rarely said aloud is that the exponents tabulated for hydrogen are
*not* the ones that fit a hydrogen atom. The published fit is made once for
$\zeta = 1$ and then rescaled,

```{math}
:label: eq-gto-scaling
\alpha_i(\zeta) = \zeta^2\,\alpha_i(1),
```

with $\zeta = 1.24$ for hydrogen, a value chosen because hydrogen in a *molecule*
has a contracted orbital. The Basis Set Exchange (Pritchard et al., *J. Chem. Inf.
Model.* **59**, 4814, 2019) distributes the scaled numbers, since molecules are what
people compute. Exercise 4 puts both sets on the free atom and finds out what the
choice costs.

### Kato's cusp, and why a Gaussian can never satisfy it

The exact eigenfunction of a Coulomb potential is not smooth at the nucleus. The
$-Z/r$ singularity must be cancelled by the kinetic term, which forces the spherical
average $\bar\psi$ of the wavefunction to obey Kato's cusp condition,

```{math}
:label: eq-gto-cusp
\left.\frac{\partial \bar\psi}{\partial r}\right|_{r \to 0} = -Z\,\bar\psi(0).
```

The hydrogen ground state $e^{-r}/\sqrt\pi$ satisfies it with slope $-1$; every
Gaussian $e^{-\alpha r^2}$ has slope $-2\alpha r \to 0$ and is *flat* at the origin.
A Gaussian basis therefore fails the cusp condition identically, no matter how many
functions are added, and the diagnostic that exposes it most sharply is the local
energy of [§6.22](../06-quantum-mechanics/variational-method.ipynb),

```{math}
:label: eq-gto-local-energy
E_L(\mathbf r) = \frac{\hat H \psi(\mathbf r)}{\psi(\mathbf r)},
```

which is *constant* for a true eigenfunction and diverges like $-Z/r$ for anything
flat at the nucleus.

### The error the basis invents

Because Gaussians are attached to nuclei, the quality of a basis at one atom depends
on what other atoms are nearby: their functions are available too, and the
variational principle will use them. Two atoms brought together are therefore each
described *better* than they were apart, and the resulting spurious attraction is
basis set superposition error. The standard diagnosis is the counterpoise
construction of Boys and Bernardi (*Mol. Phys.* **19**, 553, 1970): recompute each
fragment in the *full* basis of the pair, with the partner's nucleus and electrons
removed and only its basis functions left behind as "ghosts". For one electron and
one nucleus that construction reads

```{math}
:label: eq-gto-counterpoise
\delta_{\mathrm{BSSE}}(R)
= E\big[\text{atom in } \{\chi\}_A \cup \{\chi\}_{\mathrm{ghost}}(R)\big]
- E\big[\text{atom in } \{\chi\}_A\big] \;\le\; 0 .
```

The inequality is pure linear algebra: enlarging the space in
{eq}`eq-gto-secular` cannot raise its lowest eigenvalue. Nothing in the argument
mentions a second electron or correlation, which is exactly why the demonstration
survives all the way down to a single electron, where every other complication is
gone and the effect stands alone.

## Setup

Data and instruments only: the series palette, the exact hydrogen numbers this
notebook is judged against, the tabulated STO-3G hydrogen parameters, the lowest
Boys function, and the closed-form hydrogen $1s$ orbital of
[§6.17](../06-quantum-mechanics/hydrogen-atom.ipynb). This notebook's own machinery
is not here: the three integral matrices of {eq}`eq-gto-integrals` are written in
Exercise 1, the contraction of {eq}`eq-gto-contraction` in Exercise 3, and the
ghost-basis assembly of {eq}`eq-gto-counterpoise` in Exercise 5.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import dblquad
from scipy.linalg import eigh
from scipy.optimize import minimize_scalar
from scipy.special import erf

from ecp import draw, validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"  # data: the series palette

# data: the one-electron system this notebook never leaves, and its exact answer
Z_H = 1.0  # nuclear charge, at the origin
C_NUC = np.zeros(3)  # nuclear position, in Bohr
E_EXACT_H = -0.5  # exact hydrogen ground-state energy, in Hartree

# data: STO-3G for hydrogen as distributed by the Basis Set Exchange (Pritchard
# et al., J. Chem. Inf. Model. 59, 4814, 2019), i.e. the Hehre-Stewart-Pople 1969
# fit already scaled by zeta = 1.24 for hydrogen bound in a molecule. The
# unscaled set (the fit to a zeta = 1 Slater orbital, which is the free hydrogen
# atom) follows from it by the exponent scaling law alpha(zeta) = zeta^2 alpha(1).
STO3G_ALPHA_MOLECULE = np.array([3.42525091, 0.62391373, 0.16885540])
STO3G_COEFF = np.array([0.15432897, 0.53532814, 0.44463454])
STO3G_ZETA_H = 1.24
STO3G_ALPHA_ATOM = STO3G_ALPHA_MOLECULE / STO3G_ZETA_H**2


# instrument: the lowest Boys function of Eq. eq-gto-boys. It is a special
# function evaluation, not a method: its only content is the removable
# singularity at x = 0, where the closed form is 0/0 and the limit is 1. The
# lesson of Exercise 1 is the assembly of the matrices, not this branch.
def boys_f0(x):
    """The lowest Boys function F_0(x) = (1/2)sqrt(pi/x) erf(sqrt(x)), with F_0(0) = 1.

    What the Coulomb singularity leaves behind once the nuclear-attraction
    integral of a pair of s-Gaussians has been done over angles. It decays as
    x^(-1/2) at large argument, which is why a nucleus far from a pair of
    Gaussians contributes almost nothing to their attraction integral.

    Parameters
    ----------
    x : array_like
        Non-negative argument p |P - C|^2, dimensionless in atomic units.

    Returns
    -------
    numpy.ndarray
        F_0(x), with the x -> 0 limit taken exactly.
    """
    x = np.asarray(x, dtype=float)
    out = np.ones_like(x)
    big = x > 1e-12
    out[big] = 0.5 * np.sqrt(np.pi / x[big]) * erf(np.sqrt(x[big]))
    return out


# data: the exact hydrogen 1s orbital, handed to us in closed form by section 6.17.
# The only code here is the transcription of e^(-Zr)/sqrt(pi) onto a grid.
def hydrogen_1s(r, Z=1.0):
    """The exact normalised hydrogen 1s orbital psi(r) = sqrt(Z^3/pi) e^(-Zr).

    The reference every basis in this notebook is measured against. Its
    logarithmic derivative at the origin is exactly -Z, which is Kato's cusp
    condition, Eq. eq-gto-cusp.

    Parameters
    ----------
    r : array_like
        Radial coordinate in Bohr.
    Z : float, optional
        Nuclear charge (default 1.0).

    Returns
    -------
    numpy.ndarray
        psi(r), normalised so that the integral of |psi|^2 over space is 1.
    """
    return np.sqrt(Z**3 / np.pi) * np.exp(-Z * np.asarray(r, dtype=float))

## Exercise 1 — The three integrals, written out and certified

Everything in this notebook is one function call away once the matrices of
{eq}`eq-gto-integrals` exist, so they are built first and trusted only after they
have survived two independent tests. A basis here is a list of exponents
$\{\alpha_p\}$ together with a list of centres $\{\mathbf A_p\}$ in Bohr; the single
nucleus sits at $\mathbf C = (0,0,0)$ with charge $Z = 1$. Every matrix element
follows from the four quantities $p = \alpha_p + \alpha_q$,
$\mu = \alpha_p\alpha_q/p$, $\mathbf P = (\alpha_p\mathbf A_p + \alpha_q\mathbf
A_q)/p$ and $K = e^{-\mu|\mathbf A_p - \mathbf A_q|^2}$ of
{eq}`eq-gto-product`, all of which are full $n \times n$ arrays obtained by
`numpy` broadcasting rather than by a Python double loop.

The first test is the single-centre limit, where all three integrals collapse to
elementary expressions. For one Gaussian of exponent $\alpha$ sitting on the
nucleus, $|\mathbf A_p - \mathbf A_q| = 0$ and $|\mathbf P - \mathbf C| = 0$, so
{eq}`eq-gto-integrals` gives $S = (\pi/2\alpha)^{3/2}$, $T = \tfrac32\alpha\,S$
and $V = -\pi Z/\alpha$. The second test is a genuine two-centre pair evaluated by
two-dimensional quadrature: for spherically symmetric integrands about the nucleus
the volume element is $2\pi r^2 \sin\theta\,dr\,d\theta$, and the Laplacian needed
for the kinetic integral is available in closed form,
$\nabla^2 \chi_q = (4\alpha_q^2|\mathbf r - \mathbf A_q|^2 - 6\alpha_q)\chi_q$.

**Part a)** Write `overlap_matrix(alphas, centres)`, `kinetic_matrix(alphas,
centres)` and `nuclear_matrix(alphas, centres, Z, C)` returning the three $n \times
n$ arrays of {eq}`eq-gto-integrals`, using `numpy` broadcasting for $p$, $\mu$,
$K$ and $\mathbf P$ (the outer sum `alphas[:, None] + alphas[None, :]`, the squared
centre separations from `numpy.sum` over a broadcast difference of the centre array,
and the Setup helper `boys_f0` for $F_0$). **Write this one yourself** — the
implementation is the lesson.

**Part b)** Write `basis_energy(alphas, centres, Z=1.0, C=(0,0,0))`, which assembles
$\mathbf H = \mathbf T + \mathbf V$, solves the generalised eigenproblem $\mathbf
H\mathbf c = \varepsilon\mathbf S\mathbf c$ of {eq}`eq-gto-secular` with
`scipy.linalg.eigh(H, S)` as in
[§0.5](../00-foundations/eigenvalues-svd.ipynb), and returns the lowest eigenvalue
together with its coefficient vector.

**Part c)** Certify the single-centre limit for $\alpha = 0.5$ at the origin against
$S = (\pi)^{3/2} = 5.568328$, $T = 4.176246$ and $V = -2\pi = -6.283185$.

**Part d)** Certify a two-centre pair against `scipy.integrate.dblquad`: exponents
$\alpha_p = 3.42525091$ at $(0,0,0)$ and $\alpha_q = 0.16885540$ at $(0,0,3)$ Bohr,
with the nucleus at the origin, integrating $r$ over $[0, 20]$ Bohr and $\theta$
over $[0, \pi]$. All three off-diagonal elements must agree to twelve digits.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the machinery earns its trust

The single-centre elements must equal $(\pi/2\alpha)^{3/2} = 5.568328$,
$\tfrac32\alpha S = 4.176246$ and $-\pi Z/\alpha = -6.283185$, and the three
two-centre elements must reproduce the two-dimensional quadrature to twelve digits.

In [ ]:
validate.close(
    S_one, (np.pi / (2 * alpha_one)) ** 1.5, "single-centre overlap", rtol=1e-12
)
validate.close(T_one, 1.5 * alpha_one * S_one, "single-centre kinetic", rtol=1e-12)
validate.close(V_one, -np.pi * Z_H / alpha_one, "single-centre attraction", rtol=1e-12)
validate.close(S_pair, S_quad, "two-centre overlap vs quadrature", rtol=1e-11)
validate.close(T_pair, T_quad, "two-centre kinetic vs quadrature", rtol=1e-11)
validate.close(V_pair, V_quad, "two-centre attraction vs quadrature", rtol=1e-11)

## Exercise 2 — One Gaussian is a bad hydrogen atom

The smallest possible basis is one $s$-Gaussian sitting on the nucleus, and
{eq}`eq-gto-secular` then has a single row. Its energy is the ratio of the
one-by-one matrices, and the closed forms quoted in Exercise 1 turn it into a
function of the exponent alone,

```{math}
:label: eq-gto-one-gaussian
E(\alpha) = \frac{T + V}{S} = \frac{3}{2}\alpha - 2\sqrt{\frac{2\alpha}{\pi}} ,
```

whose minimum can be found by hand: $dE/d\alpha = 0$ at $\alpha = 8/(9\pi) =
0.2829421$, giving $E = -4/(3\pi) = -0.4244132$ Ha. Against the exact $-0.5$ that is
a 15 % error from a function which is, in every other respect, perfectly reasonable.
The reason is entirely local. A Gaussian is flat at the origin while the exact
orbital has a kink there, and {eq}`eq-gto-cusp` says the kink is not decoration
but a requirement: it is what cancels the $-Z/r$ singularity. The sharpest way to
see the failure is the local energy of {eq}`eq-gto-local-energy`, which is
exactly $-1/2$ everywhere for $e^{-r}/\sqrt\pi$ and unbounded below for any Gaussian.

**Part a)** Minimise $E(\alpha)$ over $\alpha \in [0.05, 2.0]$ with
`scipy.optimize.minimize_scalar(method="bounded")`, evaluating the objective through
the `basis_energy` you wrote in Exercise 1 (a one-function basis at the origin), and
compare the result against $\alpha^\star = 8/(9\pi)$ and $E^\star = -4/(3\pi)$ from
{eq}`eq-gto-one-gaussian`.

**Part b)** On the radial grid $r \in [10^{-3}, 6]$ Bohr with spacing $10^{-3}$ Bohr,
compute the local energy $E_L(r) = -\tfrac12 \nabla^2\psi/\psi - Z/r$ for both
$\psi = e^{-\alpha^\star r^2}$ and $\psi = e^{-r}/\sqrt\pi$, taking the radial
Laplacian $\psi'' + (2/r)\psi'$ by two applications of `numpy.gradient` rather than
from the analytic derivative. The exact orbital must return $-1/2$; the Gaussian must
diverge as $r \to 0$. Judge the constancy on $r \ge 0.05$ Bohr only, because the
$2/r$ prefactor multiplies the finite-difference error of $\psi'$ and the innermost
few points are dominated by discretisation rather than by physics.

**Part c)** Evaluate the logarithmic derivative $\psi'/\psi$ at the innermost grid
point for both functions and compare against Kato's condition {eq}`eq-gto-cusp`,
which demands $-Z = -1$; plot both orbitals and both local energies.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — the optimum, the bound, and the cusp

The optimal exponent and energy must reproduce $8/(9\pi)$ and $-4/(3\pi)$; the
energy must respect the variational bound $E > -1/2$; the finite-difference local
energy of the exact orbital must be the constant $-1/2$ Ha beyond $r = 0.05$ Bohr,
where the $2/r$ amplification of the difference error has died away; and the logarithmic
derivative at the origin must be $-1$ for the exact orbital and numerically zero
for the Gaussian, which is Kato's condition holding and failing.

In [ ]:
validate.close(
    alpha_star, 8.0 / (9.0 * np.pi), "the optimal single-Gaussian exponent", rtol=1e-6
)
validate.close(
    E_one_gaussian,
    -4.0 / (3.0 * np.pi),
    "the optimal single-Gaussian energy",
    rtol=1e-9,
)
validate.check(
    bool(E_one_gaussian > E_EXACT_H),
    "the variational bound holds for the single Gaussian",
    f"E = {E_one_gaussian:.7f} Ha > exact {E_EXACT_H}",
)
validate.close(
    EL_slater[interior],
    E_EXACT_H,
    "local energy of the exact 1s is constant",
    atol=1e-5,
)
validate.close(cusp_slater, -Z_H, "Kato's cusp condition for the exact 1s", rtol=1e-3)
validate.close(cusp_gauss, 0.0, "the Gaussian is flat at the nucleus", atol=2e-3)

## Exercise 3 — Contraction, and what freezing the coefficients costs

Three Gaussians do far better than one, and there are two entirely different ways to
use them. Left uncontracted, the three primitives are three basis functions and
{eq}`eq-gto-secular` is a $3\times3$ problem whose coefficients the variational
principle chooses. Contracted, they are welded into the single function of
{eq}`eq-gto-contraction` with the tabulated $d_i$ frozen, and the problem is
$1\times1$. The uncontracted energy must be the lower of the two, and for a reason
worth stating precisely: the contracted function is one particular vector in the
three-dimensional space the uncontracted calculation searches, namely
$c_i = d_i N_i$, so freezing the coefficients can only discard variational freedom.
That observation is also the cleanest possible test of the contraction code, because
the Rayleigh quotient of the $3\times3$ matrices evaluated at $c_i = d_i N_i$ must
equal the contracted energy to machine precision.

Chemistry contracts anyway, and not out of carelessness: an integral over contracted
functions costs the same as one over primitives once the sum has been done, so
contraction buys a smaller matrix at fixed integral count, and the frozen shape is
very nearly right in the environment the fit was made for. The environment is the
subject of Exercise 4; here we simply measure the price on the free atom, using the
exponents that the Basis Set Exchange actually distributes for hydrogen,
$\alpha = (3.42525091,\ 0.62391373,\ 0.16885540)$, with coefficients
$d = (0.15432897,\ 0.53532814,\ 0.44463454)$.

**Part a)** Solve the uncontracted $3\times3$ problem with the `basis_energy` you
wrote in Exercise 1, all three primitives on the nucleus, and report the energy
against the exact $-0.5$ Ha.

**Part b)** Write `contract(alphas, coeffs)`, returning the coefficient vector
$c_i = d_i N_i$ of {eq}`eq-gto-contraction` with $N_i = (2\alpha_i/\pi)^{3/4}$,
and obtain the contracted energy as the Rayleigh quotient
$(\mathbf c^{\mathsf T}(\mathbf T + \mathbf V)\mathbf c)/(\mathbf c^{\mathsf T}
\mathbf S\mathbf c)$ formed from the same three matrices with the `@` product.
Confirm that $\mathbf c^{\mathsf T}\mathbf S\mathbf c = 1$, which is the statement
that the tabulated $d_i$ are normalised.

**Part c)** Report the difference between the two energies, and plot the three
weighted primitives $d_i N_i e^{-\alpha_i r^2}$, their contracted sum, and the exact
$e^{-r}/\sqrt\pi$ on $r \in [0, 4]$ Bohr.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3 — the contraction sits inside the uncontracted space

The contracted function must be normalised, its energy must equal the Rayleigh
quotient of the $3\times3$ problem at $c_i = d_i N_i$ to machine precision, and it
must lie strictly above the uncontracted energy, which in turn lies strictly above
the exact $-1/2$ Ha. The ordering is the variational principle, and it is what makes
the two numbers comparable at all.

In [ ]:
validate.close(
    norm_mol, 1.0, "the tabulated STO-3G contraction is normalised", rtol=1e-7
)
validate.close(
    E_uncontracted_mol, -0.49574080, "uncontracted STO-3G hydrogen energy", rtol=1e-7
)
validate.close(
    E_contracted_mol, -0.46658185, "contracted STO-3G hydrogen energy", rtol=1e-7
)
validate.check(
    bool(E_uncontracted_mol < E_contracted_mol < E_EXACT_H + 1.0),
    "freezing the coefficients raises the energy, and both stay above the exact value",
    f"{E_uncontracted_mol:.8f} < {E_contracted_mol:.8f}, both > {E_EXACT_H}",
)
validate.check(
    bool(E_uncontracted_mol > E_EXACT_H and E_contracted_mol > E_EXACT_H),
    "the variational bound holds for both three-Gaussian calculations",
    f"uncontracted {E_uncontracted_mol:.8f}, contracted {E_contracted_mol:.8f}, exact {E_EXACT_H}",
)

## Exercise 4 — A basis optimised for one environment is wrong in another

The contraction of Exercise 3 cost $0.029$ Ha, which is enormous: it is seven tenths
of an electronvolt, and it is larger than the correlation energy of helium computed
in [§8.1](many-electron-problem.ipynb). Before blaming contraction as such, it is
worth asking *which* Slater orbital those three Gaussians were fitted to.
{eq}`eq-gto-scaling` is the answer: the published fit is made once for $\zeta = 1$
and then rescaled, and the numbers distributed for hydrogen carry $\zeta = 1.24$,
because hydrogen in a molecule has a contracted orbital. The free atom is a
$\zeta = 1$ system, so the shipped basis is fitted to the wrong atom.

Undoing the scaling is one division: $\alpha_i(1) = \alpha_i(1.24)/1.24^2$, giving
$\alpha = (2.22766058,\ 0.40577116,\ 0.10981751)$. Running the free hydrogen atom in
both sets, contracted and uncontracted, separates two effects that Exercise 3 could
not tell apart. The result is worth predicting before computing: the contracted
comparison behaves exactly as advertised, and the uncontracted one does not.

**Part a)** Compute the contracted energy in both exponent sets, using the
`contract` helper of Exercise 3 and the same Rayleigh quotient, and report the
difference.

**Part b)** Compute the uncontracted energy in both exponent sets with
`basis_energy`, and report that difference too. State which of the two comparisons
carries the penalty, and therefore where in the construction the environment
dependence actually lives.

**Part c)** Quantify the shapes rather than only the energies: compute the overlap
$\langle \phi | 1s\rangle$ of each *normalised* contracted function with the exact
$e^{-r}/\sqrt\pi$, as $\sum_i c_i \int e^{-\alpha_i r^2} e^{-r}/\sqrt\pi\,d^3r$ with
each radial integral evaluated by `scipy.integrate.dblquad` over $r \in [0, 40]$
Bohr and $\theta \in [0, \pi]$, then plot both contracted functions against the exact
orbital.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — the penalty, and where it does not live

In the contracted comparison the free-atom exponents must win, and by a margin of
about $0.028$ Ha; in the uncontracted comparison the ordering must *reverse*, which
is the honest surprise of this exercise and the evidence that the environment
dependence of a minimal basis is carried by the frozen coefficients rather than by
the exponents alone. The free-atom contracted function must also overlap the exact
orbital better than the molecular one, above $0.999$.

In [ ]:
validate.close(
    E_contracted_atom, -0.49490709, "free-atom-scaled contracted energy", rtol=1e-7
)
validate.close(
    E_uncontracted_atom, -0.49501040, "free-atom-scaled uncontracted energy", rtol=1e-7
)
validate.check(
    bool(E_contracted_atom < E_contracted_mol),
    "contracted: the free-atom exponents beat the molecular ones",
    f"{E_contracted_atom:.8f} Ha < {E_contracted_mol:.8f} Ha, margin {E_contracted_mol - E_contracted_atom:.5f} Ha",
)
validate.check(
    bool(E_uncontracted_mol < E_uncontracted_atom),
    "uncontracted: the ordering reverses, so the penalty is the frozen coefficients",
    f"{E_uncontracted_mol:.8f} Ha < {E_uncontracted_atom:.8f} Ha, margin {E_uncontracted_atom - E_uncontracted_mol:.2e} Ha",
)
validate.check(
    bool(ov_atom > 0.999 > ov_mol),
    "the free-atom contraction is the better shape as well as the better energy",
    f"overlaps: free-atom {ov_atom:.6f}, molecular {ov_mol:.6f}",
)

## Exercise 5 — The error the basis invents, on one electron

Here is the experiment. A hydrogen atom sits at the origin: one proton, one electron,
nothing else in the universe. Its exact energy is $-1/2$ Ha and its computed energy
in the three uncontracted primitives of Exercise 3 is $-0.49574080$ Ha. Now place
three more Gaussians, with the same three exponents, at $(0,0,R)$ Bohr. No nucleus
goes with them and no second electron: they are pure basis functions, and the
Hamiltonian is unchanged. These are the *ghost* functions of the counterpoise
construction, {eq}`eq-gto-counterpoise`. The energy of the hydrogen atom will
drop, and it will drop by an amount that depends on $R$, which is the whole
pathology of an atom-centred basis in its simplest possible form.

It is worth being explicit about what is not going on. Nothing here is binding: there
is no second nucleus for the electron to be attracted to, no second electron to
correlate with, and no dispersion, because dispersion is a two-electron effect and
there is only one electron. The lowering is purely the statement that a bigger
expansion space in {eq}`eq-gto-secular` cannot give a higher lowest eigenvalue.
In a real dimer calculation exactly this lowering is mistaken for chemistry, and the
counterpoise recipe of Boys and Bernardi subtracts it by computing each monomer in
the dimer basis, which is precisely the calculation below.

Basis set superposition error is normally taught on a bound dimer with a
quantum-chemistry package, and we found no published teaching treatment of it on a
single electron. This exercise is therefore doing something the teaching literature
does not; the physics is nonetheless standard, because the counterpoise argument is
purely variational and never invokes correlation.

The shape of the curve is the lesson, and it is not monotonic. Very close in, the
ghosts are nearly linearly dependent on the real functions, so they add little
genuine freedom and the overlap matrix pays for what little they add. Very far out
they are irrelevant. In between they are both maximally helpful and maximally
dishonest.

**Part a)** Write `ghost_energy(R, alphas)`, which assembles a basis of $2n$
functions with exponents `numpy.concatenate([alphas, alphas])`, the first $n$ centred
at the origin and the last $n$ at $(0,0,R)$, keeps the single nucleus $Z = 1$ at the
origin, and returns the lowest energy from `basis_energy` together with the smallest
eigenvalue of the *normalised* overlap matrix $\tilde S_{pq} = S_{pq}/\sqrt{S_{pp}
S_{qq}}$ obtained from `numpy.linalg.eigvalsh`. **Write this one yourself** — the
implementation is the lesson.

**Part b)** Tabulate the lowering $\delta_{\mathrm{BSSE}}(R)$ of
{eq}`eq-gto-counterpoise` at $R = 2, 3, 4, 8, 12$ Bohr against the monomer-basis
energy $-0.49574080$ Ha, using the molecular STO-3G exponents uncontracted.

**Part c)** Sweep $R$ over $[0.5, 12]$ Bohr in steps of $0.05$ Bohr, locate the
maximum lowering with `scipy.optimize.minimize_scalar(method="bounded")` on
$[2.5, 5.0]$, and plot both the lowering and the smallest normalised-overlap
eigenvalue against $R$. Compare the size of the largest lowering with the basis
incompleteness error $-0.5 - (-0.49574080)$, and say which of the two errors is
which.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — the lowering, its sign, its shape, and its size

Four independent facts. The energy with ghosts can never exceed the monomer energy,
at any $R$, because the expansion space only grew. At $R = 12$ Bohr the two must
agree to better than $10^{-9}$ Ha, since the ghosts have nothing left to contribute.
The maximum lowering must sit strictly inside the scan rather than at either end,
and must exceed the lowering both at $R = 1$ and at $R = 8$ Bohr, which is the
non-monotonicity stated as a testable claim. And the whole effect must remain a small
fraction of the incompleteness error, so that the two errors cannot be confused.

In [ ]:
validate.check(
    bool(np.all(lowering <= 1e-14)),
    "adding ghost functions never raises the energy, at any R",
    f"largest observed rise = {lowering.max():.2e} Ha over {len(R_scan)} distances",
)
E_far, _ = ghost_energy(12.0, STO3G_ALPHA_MOLECULE, Z_H)
validate.close(E_far, E_monomer, "the ghosts stop mattering at R = 12 Bohr", atol=1e-9)
validate.close(low_peak, -1.7753e-4, "the largest superposition lowering", rtol=1e-3)
validate.close(R_peak, 3.542, "the distance at which the ghosts help most", rtol=1e-2)
low_near, _ = ghost_energy(1.0, STO3G_ALPHA_MOLECULE, Z_H)
low_out, _ = ghost_energy(8.0, STO3G_ALPHA_MOLECULE, Z_H)
validate.check(
    bool(low_peak < low_near - E_monomer and low_peak < low_out - E_monomer),
    "the lowering is non-monotonic: an interior maximum, not a decaying tail",
    f"delta(1.0) = {low_near - E_monomer:.2e}, delta({R_peak:.2f}) = {low_peak:.2e}, delta(8.0) = {low_out - E_monomer:.2e} Ha",
)
validate.check(
    bool(abs(low_peak) < 0.05 * abs(incompleteness)),
    "superposition error stays far below basis incompleteness error",
    f"{abs(low_peak):.3e} Ha vs {abs(incompleteness):.3e} Ha",
)
validate.check(
    bool(smin_scan[0] < 0.1 * smin_monomer),
    "the overlap matrix becomes near-singular as the ghosts close in",
    f"smallest normalised eigenvalue {smin_scan[0]:.4f} at R = 0.5 Bohr vs {smin_monomer:.4f} for the monomer basis",
)

## Exercise 6 — Superposition error is a symptom of incompleteness

If the ghosts help only because the atom's own basis is not good enough, then a
better basis must leave them less to do, and in the limit of a complete basis the
lowering must vanish entirely. That is a prediction, and it is testable with a family
of bases that can be systematically enlarged. Even-tempered sets are the standard
device: exponents in geometric progression,

```{math}
:label: eq-gto-even-tempered
\alpha_k = \alpha_c\,\beta^{\,k - (N-1)/2},
\qquad k = 0, 1, \dots, N-1,
```

which for fixed $\alpha_c$ and $\beta$ grows outward in both directions as $N$
increases, adding tighter functions to resolve the nucleus and more diffuse ones to
resolve the tail. Reeves introduced them in 1963 and they remain the usual way to
approach a Gaussian basis-set limit; the ratio $\beta$ is chosen by hand, and here we
fix $\alpha_c = 1$ and $\beta = 3$ so that $N$ is the only thing varying.

**Part a)** For $N = 1, \dots, 6$, build the exponents of
{eq}`eq-gto-even-tempered` with `numpy.arange`, put all $N$ functions on the nucleus,
and compute the energy with `basis_energy`. Report the incompleteness error
$E(N) - (-1/2)$.

**Part b)** For each $N$, run the ghost sweep of Exercise 5 with these exponents over
$R \in [1, 12]$ Bohr in steps of $0.25$ Bohr, using the `ghost_energy` you wrote
there, and record the largest lowering.

**Part c)** Plot both errors against $N$ on a logarithmic axis and state the
relationship between them. Note in particular whether the superposition error is
ever the larger of the two.

:::{admonition} With your assistant
:class: tip
The even-tempered exponent generator of {eq}`eq-gto-even-tempered` is a two-line
function with an off-by-one waiting in the exponent offset, and exactly the kind of
thing an assistant writes quickly. Have it produce one, then run the check that is
yours: the sequence $E(1) > E(2) > \dots > E(6)$ must be strictly decreasing, because
with $\alpha_c$ and $\beta$ fixed the $N$-function set is a genuine subset of the
$(N+1)$-function set for odd-to-odd and even-to-even steps, and adding functions to a
variational calculation can never raise its energy. The check is yours.
:::

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — both errors fall, and one always leads

The energies must decrease strictly with $N$, since the family is nested at fixed
$\alpha_c$ and $\beta$. Both error measures must fall monotonically, and the
superposition error must lie below the incompleteness error at every $N$: a basis
cannot invent more error than it is missing.

In [ ]:
validate.check(
    bool(np.all(np.diff(et_energies) < 0)),
    "the even-tempered energies decrease strictly with basis size",
    f"energies {np.array2string(et_energies, precision=6)}",
)
validate.check(
    bool(np.all(np.diff(et_error) < 0)),
    "the incompleteness error falls monotonically",
    f"{np.array2string(et_error, precision=3, formatter={'float_kind': lambda v: f'{v:.2e}'})}",
)
validate.check(
    bool(np.all(np.diff(np.abs(et_bsse)) < 0)),
    "the superposition error falls monotonically with basis quality",
    f"{np.array2string(np.abs(et_bsse), formatter={'float_kind': lambda v: f'{v:.2e}'})}",
)
validate.check(
    bool(np.all(np.abs(et_bsse) < et_error)),
    "superposition error never exceeds the incompleteness it comes from",
    f"largest ratio = {np.max(np.abs(et_bsse) / et_error):.3f}",
)

## Exercise 7 — What makes a basis good, measured against the plane-wave contract

[§8.10](plane-waves-pseudopotentials.ipynb) stated a standard for what a basis
should offer, and it is worth quoting in substance: systematic completeness with one
knob, the cutoff, and unbiased coverage of space. That standard has three parts, and
the Gaussian basis satisfies none of them cleanly. There is no single knob, because
a Gaussian basis is specified by exponents, contraction patterns, angular momenta and
a choice of family. There is no ordering of the standard families by size, because
they are separately optimised objects rather than truncations of one sequence. And
there is no unbiased coverage, because every function is attached to a nucleus, which
is exactly why Exercise 5 had anything to show.

The middle claim is the one we can measure here. Exercise 6 built a family in which
size and quality do move together, so the failure is not that Gaussian bases are
incapable of systematic improvement; it is that improvement is guaranteed only
*within* one family. Put the STO-3G primitives of Exercise 3 on the same axis and the
guarantee disappears: three well-chosen exponents beat four evenly-tempered ones.
Nothing comparable can happen with a plane-wave cutoff, where a larger basis is
literally a superset of a smaller one.

The other two claims are stated rather than computed, and the reason is worth being
explicit about. Polarisation functions ($p$ functions on hydrogen, $d$ on carbon) let
a spherical atom deform in a bond, and diffuse functions carry anions and Rydberg
states, but both need general angular momentum, which is outside the one-electron
$s$-only machinery this notebook built. We do not invent an exercise for them and do
not imply that a canonical teaching demonstration exists.

**Part a)** Assemble the comparison: the even-tempered energies $E(N)$ from Exercise
6, the uncontracted three-primitive STO-3G energy from Exercise 3, and the contracted
STO-3G energy from Exercise 3, plotted against the number of *variational* functions
(three, three and one respectively) with the exact $-1/2$ Ha marked.

**Part b)** Report which even-tempered basis size the three uncontracted STO-3G
functions beat, and which one beats them, and state what that ordering does to the
idea that basis size measures basis quality.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7 — a smaller basis winning

The claim to check is the uncomfortable one: a three-function basis strictly below a
four-function basis of a different family, with both still above the exact energy.
If that ordering held for plane waves it would be a bug; here it is the point.

In [ ]:
validate.check(
    bool(E_uncontracted_mol < et_energies[3]),
    "three STO-3G primitives beat the four-function even-tempered basis",
    f"{E_uncontracted_mol:.8f} Ha < {et_energies[3]:.8f} Ha",
)
validate.check(
    bool(E_uncontracted_mol > et_energies[4]),
    "and are beaten by the five-function even-tempered basis",
    f"{E_uncontracted_mol:.8f} Ha > {et_energies[4]:.8f} Ha",
)
validate.check(
    bool(np.all(et_energies > E_EXACT_H) and E_uncontracted_mol > E_EXACT_H),
    "every energy on the comparison axis is a genuine variational upper bound",
    f"lowest value on the axis = {min(et_energies.min(), E_uncontracted_mol):.8f} Ha > {E_EXACT_H}",
)

## Notebook summary

Three closed forms and an error function were enough. The overlap, kinetic and
nuclear-attraction integrals of {eq}`eq-gto-integrals` were assembled by
broadcasting and certified twice: against the single-centre limits
$(\pi/2\alpha)^{3/2}$, $\tfrac32\alpha S$ and $-\pi Z/\alpha$, and against
two-dimensional quadrature for a two-centre pair, agreeing to twelve digits. With
them, the generalised eigenproblem $\mathbf H\mathbf c = \varepsilon\mathbf S\mathbf
c$ of [§0.5](../00-foundations/eigenvalues-svd.ipynb) is the whole method.

The best single $s$-Gaussian gives $\alpha^\star = 8/(9\pi) = 0.282942$ and
$E = -4/(3\pi) = -0.424413$ Ha, missing the exact $-0.5$ by 15 %, and the reason was
made visible rather than asserted: the finite-difference local energy is the constant
$-0.5$ Ha for $e^{-r}/\sqrt\pi$ and diverges like $-1/r$ for the Gaussian, whose
logarithmic derivative at the origin is $0$ instead of Kato's $-Z = -1$. Three
primitives left uncontracted reach $-0.49574080$ Ha; the same three frozen into the
STO-3G contraction reach only $-0.46658185$ Ha, and the contracted energy was shown
to be exactly the Rayleigh quotient of the $3\times3$ problem at $c_i = d_i N_i$,
which is why the ordering is guaranteed. Undoing the $\zeta = 1.24$ molecular scaling
recovers $-0.49490709$ Ha contracted, a gain of $0.028$ Ha, while the uncontracted
comparison *reverses*: the environment dependence of a minimal basis lives in the
frozen coefficients, not in the exponents alone.

The centrepiece was a hydrogen atom given three basis functions with no nucleus
behind them. Its energy fell, by $1.78\times10^{-4}$ Ha at most, with the maximum at
$R = 3.54$ Bohr rather than at contact: close in, the ghosts are nearly linearly
dependent on the real functions and the smallest normalised overlap eigenvalue
collapses from $0.162$ to $0.011$; far out they are irrelevant, and by $R = 12$ Bohr
the lowering is $7\times10^{-11}$ Ha. That number is 4 % of the basis incompleteness
error $4.26\times10^{-3}$ Ha, and telling the two apart is the practical skill.
Enlarging an even-tempered basis from $N = 1$ to $N = 6$ drove both errors down by
more than three orders of magnitude together, confirming that the invented error is a
symptom of the missing one. And three STO-3G primitives beat a four-function
even-tempered basis while losing to a five-function one, so basis size does not order
basis quality across families, which is precisely the guarantee the plane-wave cutoff
of [§8.10](plane-waves-pseudopotentials.ipynb) does give.

## Outlook

- The counterpoise correction as chemistry actually uses it subtracts
  $\delta_{\mathrm{BSSE}}$ from a *binding* energy, monomer by monomer, on a real
  dimer. That calculation needs two-electron integrals and a self-consistent field,
  so it needs a quantum-chemistry package; the argument, though, is the one verified
  here on one electron, unchanged.
- Polarisation and diffuse functions are the two extensions a minimal basis most
  obviously lacks, and both require general angular momentum. The correlation-consistent
  families of Dunning arrange them into a sequence designed to be extrapolated toward
  the basis-set limit, which is the nearest thing a Gaussian basis has to the single
  knob of [§8.10](plane-waves-pseudopotentials.ipynb).
- The pathology met here is one reason plane-wave codes and Gaussian codes coexist
  rather than one displacing the other. An atom-centred basis is compact and follows
  the atoms, at the price of an error that follows them too; the plane-wave basis of
  [§8.11](epm-band-structures.ipynb) is attached to the cell instead, cannot have this
  error at all, and pays for that with the core-region cost measured in
  [§8.10](plane-waves-pseudopotentials.ipynb).
- Everything here was one electron in a fixed external potential. Put two electrons
  in the same basis and the two-electron integrals appear, four indices at a time,
  and with them the whole machinery of [§8.3](hartree-fock-atoms.ipynb) in its
  Roothaan form: the same {eq}`eq-gto-secular`, with a Hamiltonian that now
  depends on its own solution.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()